In [2]:
import os
import numpy as np
import pandas as pd


def clean_uber_data(
    input_path: str = r"C:\Users\HP-PC\Desktop\Project_X\Data\raw_booking.csv",
    output_path: str = r"C:\Users\HP-PC\Desktop\Project_X\Data\cleaned_booking.csv",
):
    print(f"--> [Step 1] Loading raw dataset from: {input_path}")
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Source file not found at: {input_path}")

    df = pd.read_csv(input_path)
    print(f"    Raw records loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

    # 1. Standardize column names to lower_snake_case
    print("--> [Step 2] Formatting column headers...")
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("cancelled_rides_by_", "is_cancelled_by_")
    )
    df.rename(columns={"incomplete_rides": "is_incomplete"}, inplace=True)

    # 2. Clean string columns (strip whitespace, outer quotes, clean null representations)
    print("--> [Step 3] Sanitizing text columns...")
    str_cols = df.select_dtypes(include=["object","string"]).columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip().str.replace('"', "")

    # Replace string 'nan', 'None', and empty strings with genuine np.nan
    df.replace({"nan": np.nan, "None": np.nan, "": np.nan}, inplace=True)

    # 3. Standardize Date and Time
    print("--> [Step 4] Formatting Date and Time...")
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    if "time" in df.columns:
        df["time"] = pd.to_datetime(
            df["time"], format="%H:%M:%S", errors="coerce"
        ).dt.strftime("%H:%M:%S")

    # 4. Fill flag indicators with 0 and cast to int (0 or 1)
    print("--> [Step 5] Converting status flags to binary integers...")
    flag_cols = [
        "is_cancelled_by_customer",
        "is_cancelled_by_driver",
        "is_incomplete",
    ]
    for col in flag_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    # 5. Fill categorical missing values with 'Not Applicable'
    print("--> [Step 6] Categorizing missing reason fields...")
    cat_cols = [
        "reason_for_cancelling_by_customer",
        "driver_cancellation_reason",
        "incomplete_rides_reason",
        "payment_method",
    ]
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].fillna("Not Applicable")

    # 6. Explicitly enforce numeric floats for metrics (keeping NaN where appropriate)
    print("--> [Step 7] Validating operational and financial metrics...")
    numeric_cols = [
        "avg_vtat",
        "avg_ctat",
        "booking_value",
        "ride_distance",
        "driver_ratings",
        "customer_rating",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 7. Summary Audit
    print("\n================ DATA CLEANING AUDIT ================")
    print(f"Total Rows:     {df.shape[0]:,}")
    print(f"Total Columns:  {df.shape[1]}")
    completed_rides = (df["booking_status"] == "Completed").sum()
    completed_rev = df[df["booking_status"] == "Completed"][
        "booking_value"
    ].sum()
    print(f"Completed Rides:     {completed_rides:,}")
    print(f"Gross Completed Rev: ₹{completed_rev:,.2f}")
    print("=====================================================\n")

    # Save output CSV
    df.to_csv(output_path, index=False)
    print(f"SUCCESS: Cleaned dataset saved to '{output_path}'.")
    return df


if __name__ == "__main__":
    clean_uber_data()

--> [Step 1] Loading raw dataset from: C:\Users\HP-PC\Desktop\Project_X\Data\raw_booking.csv
    Raw records loaded: 150,000 rows, 21 columns
--> [Step 2] Formatting column headers...
--> [Step 3] Sanitizing text columns...
--> [Step 4] Formatting Date and Time...
--> [Step 5] Converting status flags to binary integers...
--> [Step 6] Categorizing missing reason fields...
--> [Step 7] Validating operational and financial metrics...

================ DATA CLEANING AUDIT ================
Total Rows:     150,000
Total Columns:  21
Completed Rides:     93,000
Gross Completed Rev: ₹47,260,574.00

SUCCESS: Cleaned dataset saved to 'C:\Users\HP-PC\Desktop\Project_X\Data\cleaned_booking.csv'.
